# 05 · Race results from Jolpica (2018–2025)

Finishing order, grid position, status (finished / lapped / retirement reason) and points for every race, from the free [Jolpica](https://github.com/jolpica/jolpica-f1) API (the successor to Ergast; no login needed). About five requests per season, so one run covers everything.

These are the targets and key inputs for the baseline finishing-position model. Run notebook 03 first so the cross-check against lap data works.

In [ ]:
# @title Setup: mount Drive, get the latest code, install FastF1
REPO_URL = "https://github.com/YOUR_USERNAME/f1-race-sim.git"  # @param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/f1sim_data"  # @param {type:"string"}

import os, sys
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "fastf1>=3.9,<4"

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["F1SIM_DATA_DIR"] = DATA_DIR

%load_ext autoreload
%autoreload 2
print("Code:", REPO_DIR, "| Data:", DATA_DIR)

In [ ]:
# @title Download (one run is enough)
START_YEAR = 2018  # @param {type:"integer"}
END_YEAR = 2025  # @param {type:"integer"}

from f1sim.paths import get_paths
from f1sim.ingest_results import ingest_results_seasons

paths = get_paths()
summary = ingest_results_seasons(range(START_YEAR, END_YEAR + 1), paths)

In [ ]:
# @title Coverage per season
from f1sim.ingest_results import load_results

results = load_results(paths)
per_race = results.groupby(["Year", "Round"]).agg(
    Drivers=("DriverId", "size"),
    Classified=("ClassifiedPosition", lambda s: int(s.str.isdigit().sum())),
    PitLaneStarts=("GridPosition", lambda s: int((s == 0).sum())),
).reset_index()
per_race["NotClassified"] = per_race["Drivers"] - per_race["Classified"]
per_race.groupby("Year").agg(
    Races=("Round", "size"), MinDrivers=("Drivers", "min"), MaxDrivers=("Drivers", "max"),
    NotClassified=("NotClassified", "sum"), PitLaneStarts=("PitLaneStarts", "sum"),
)

In [ ]:
# @title Cross-check: do results and lap data cover the same races, with the same names?
import pandas as pd
from f1sim.ingest_tracing import load_race_ti

lap_races = load_race_ti("laps", paths)[["Year", "Round", "Event"]].drop_duplicates()
res_races = results[["Year", "Round", "Event"]].drop_duplicates()
both = lap_races.merge(res_races, on=["Year", "Round"], how="outer",
                       suffixes=("Laps", "Results"), indicator=True)

only_one = both[both["_merge"] != "both"]
name_diff = both[(both["_merge"] == "both") & (both["EventLaps"] != both["EventResults"])]
print(f"Races in both: {(both['_merge'] == 'both').sum()} | only in one source: {len(only_one)} | "
      f"same round but different name: {len(name_diff)}")
pd.concat([only_one, name_diff])

In [ ]:
# @title Most common statuses (finished, lapped, retirement reasons)
results["Status"].value_counts().head(25)